# EcoData | Organização e Qualidade de Dados Ambientais

**Python, Automação e Pandas.**

## 1. Bibliotecas utilizadas

Esta etapa importa as ferramentas necessárias ao processamento:

- **pathlib:** manipulação de caminhos, arquivos e diretórios.
- **pandas:** leitura, conversão e análise de dados em CSV e Excel.
- **hashlib:** cálculo de assinaturas SHA-256 para identificar arquivos idênticos.
- **shutil:** movimentação de arquivos para a pasta de arquivamento.

**Nenhum arquivo é modificado nesta etapa.**

In [28]:
from pathlib import Path
import pandas as pd
import hashlib
import shutil

## 2. Configuração dos diretórios

O notebook encontra o projeto a partir da pasta de execução (EcoData ou sua pasta pai) e define:

- `dados_recebidos`: originais CSV/XLSX, preservados.
- `EcoData_Organizado`: CSVs convertidos e classificados.
- `EcoData_Duplicados_Arquivados`: cópias redundantes arquivadas por execução explícita do tratamento.

Esta célula apenas mostra os caminhos e exige que `dados_recebidos` exista. O script separado `monitorar_ingestao.py` ainda aponta para `Automação_Ecodata.ipynb`, removido intencionalmente do disco. Até que o script seja atualizado, ele não consegue carregar o organizador; abrir este notebook RV01/RV02 não inicia o monitor.


In [29]:
pasta_execucao = Path.cwd().resolve()

if (pasta_execucao / "dados_recebidos").is_dir():
    pasta_projeto = pasta_execucao
else:
    projetos_candidatos = [
        item for item in pasta_execucao.iterdir()
        if item.is_dir() and (item / "dados_recebidos").is_dir()
    ]

    if len(projetos_candidatos) != 1:
        raise FileNotFoundError(
            "Não foi possível identificar uma única pasta do EcoData a partir de "
            f"{pasta_execucao}. Execute o notebook na pasta do projeto ou em sua pasta pai."
        )

    pasta_projeto = projetos_candidatos[0].resolve()

pasta_origem = pasta_projeto / "dados_recebidos"
pasta_destino = pasta_projeto / "EcoData_Organizado"

if not pasta_origem.is_dir():
    raise FileNotFoundError(f"Pasta de origem não encontrada: {pasta_origem}")

print(f"Pasta do projeto: {pasta_projeto}")
print(f"Origem: {pasta_origem}")
print(f"Destino: {pasta_destino}")


Pasta do projeto: C:\Users\giova\Documents\Python\Projetos - Pandas, ML e Programas\6 - Automação e Organização de Dados Ambientais
Origem: C:\Users\giova\Documents\Python\Projetos - Pandas, ML e Programas\6 - Automação e Organização de Dados Ambientais\dados_recebidos
Destino: C:\Users\giova\Documents\Python\Projetos - Pandas, ML e Programas\6 - Automação e Organização de Dados Ambientais\EcoData_Organizado


## 3. Leitura e classificação dos arquivos

Nesta etapa, o programa define duas funções auxiliares:

**`listar_arquivos()`**

Percorre a pasta de origem e identifica arquivos nos formatos CSV e XLSX.

**`identificar_categoria()`**

Analisa as palavras presentes nos nomes dos arquivos e determina sua categoria:

| Categoria | Identificadores |
|---|---|
| Chuvas | chuva, precipitacao, pluviometria |
| Temperatura | temperatura, temp, termometria |
| Qualidade do Ar | ar, poluicao, pm25; qualidade_ar contém o token ar |
| Pendências | Arquivos não reconhecidos ou com pistas conflitantes |

A classificação utiliza tokens completos do nome, normalizando maiúsculas/minúsculas, hífens e underscores. A palavra `qualidade` isolada não identifica qualidade do ar. Pistas de várias categorias, ou de ar junto de água/solo, seguem para `Pendencias` com aviso. O conteúdo interno não é usado para escolher a categoria.

**Importante:** essas funções apenas identificam informações. Não movem nem alteram os documentos.

In [30]:
def listar_arquivos(pasta_origem):
    pasta_origem = Path(pasta_origem)

    if not pasta_origem.is_dir():
        raise FileNotFoundError(
            f"Pasta de origem não encontrada: {pasta_origem}"
        )

    arquivos = []

    for arquivo in pasta_origem.iterdir():
        if arquivo.is_file() and arquivo.suffix.lower() in (".csv", ".xlsx"):
            arquivos.append(arquivo)

    return sorted(arquivos)

def identificar_categoria(arquivo):
    nome = arquivo.stem.lower().replace("-", "_")
    palavras = set(nome.split("_"))

    indicios = {
        "Chuvas": {"chuva", "precipitacao", "pluviometria"},
        "Temperatura": {"temperatura", "temp", "termometria"},
        "Qualidade_do_Ar": {"ar", "poluicao", "pm25"}
    }
    categorias = [
        categoria for categoria, termos in indicios.items()
        if palavras.intersection(termos)
    ]

    # Água e solo não possuem categoria no contrato atual. A palavra genérica
    # "qualidade" nunca basta; pistas de ar junto desses temas exigem revisão.
    outro_tema = (
        "Qualidade_do_Ar" in categorias
        and bool(palavras.intersection({"agua", "solo"}))
    )
    if len(categorias) > 1 or outro_tema:
        motivo = (
            "pistas de mais de uma categoria: " + ", ".join(categorias)
            if len(categorias) > 1
            else "pistas de qualidade do ar junto de água ou solo"
        )
        print(
            f"[CLASSIFICAÇÃO AMBÍGUA] {arquivo.name}: {motivo}. "
            "Encaminhado para Pendencias; confira o nome original."
        )
        return "Pendencias"

    return categorias[0] if categorias else "Pendencias"


## 4. Cálculo de hash, conversão e organização

`calcular_hash()` usa SHA-256 dos bytes integrais para detectar arquivos binariamente idênticos. `converter_para_csv()` lê CSV ou a primeira aba de XLSX e grava UTF-8-SIG/CRLF, sem índice. `id_registro` e `estacao_id` são textuais; as demais colunas seguem a inferência do Pandas. IDs Excel numericamente formatados, datas ou fórmulas exigem conferência; a conversão não inventa zeros.

`organizar_arquivos()` classifica Chuvas, Temperatura, Qualidade_do_Ar e Pendencias. Compara o CSV esperado com destinos existentes por uma assinatura dos **campos CSV**, normalizando LF/CRLF e preservando diferenças textuais. Ela não equivale ao hash binário de `calcular_hash()`. A função prepara um temporário, publica sem sobrescrever, ignora conteúdo já presente e preserva conteúdo divergente com nome alternativo e aviso de conflito. Um conteúdo arquivado só deixa de ser importado se houver cópia ativa válida; sem ela, a recuperação é permitida. Possível diferença histórica apenas numérica nos IDs gera **migração R5 pendente**, sem publicação automática. Uma cópia ativa em outra categoria gera aviso de migração de categoria.

**Resultado salvo da reexecução com o inventário de 24/10/2026:** 48 arquivos recebidos foram ignorados porque já estavam válidos ou redundantes; zero publicações, conflitos, migrações R5 pendentes e erros. O resumo é calculado a cada execução, não contém números fixos.


In [31]:
# ==================================================
# CÁLCULO DE HASH SHA-256
# ==================================================

def calcular_hash(arquivo):
    hash_arquivo = hashlib.sha256()

    with open(arquivo, "rb") as arquivo_aberto:
        for bloco in iter(
            lambda: arquivo_aberto.read(8192),
            b""
        ):
            hash_arquivo.update(bloco)

    return hash_arquivo.hexdigest()


# ==================================================
# CONVERSÃO DE ARQUIVOS PARA CSV
# ==================================================

def converter_para_csv(arquivo, destino):
    arquivo = Path(arquivo)
    # Esquema observado: somente os identificadores são textuais.
    tipos_identificadores = {"id_registro": "string", "estacao_id": "string"}

    if arquivo.suffix.lower() == ".csv":
        dados = pd.read_csv(
            arquivo,
            encoding="utf-8-sig",
            dtype=tipos_identificadores
        )

    else:
        # Pandas usa openpyxl para XLSX. Conferimos a representação armazenada
        # antes de converter: um número formatado como 000 não é texto "001".
        from openpyxl import load_workbook

        planilha = load_workbook(arquivo, read_only=True, data_only=False)
        try:
            aba = planilha.worksheets[0]
            cabecalho = next(aba.iter_rows(min_row=1, max_row=1), ())
            identificadores = {
                indice: celula.value
                for indice, celula in enumerate(cabecalho)
                if celula.value in tipos_identificadores
            }
            for linha in aba.iter_rows(min_row=2):
                for indice, nome in identificadores.items():
                    celula = linha[indice]
                    if celula.value is None:
                        continue
                    formato_ambiguo = (
                        celula.data_type == "n"
                        and celula.number_format not in ("General", "0")
                    )
                    if formato_ambiguo or celula.data_type in ("d", "f"):
                        raise ValueError(
                            f"IDENTIFICADOR AMBÍGUO: {arquivo.name}, "
                            f"{aba.title}!{celula.coordinate}, coluna {nome}; "
                            f"tipo {celula.data_type}, formato {celula.number_format!r}. "
                            "Confirme o esquema e prepare uma cópia com o ID textual; "
                            "nenhum zero será inventado."
                        )
        finally:
            planilha.close()

        dados = pd.read_excel(arquivo, dtype=tipos_identificadores)

    dados.to_csv(
        destino,
        index=False,
        encoding="utf-8-sig",
        lineterminator="\r\n"
    )


# ==================================================
# ORGANIZAÇÃO DOS ARQUIVOS
# ==================================================

def organizar_arquivos(pasta_origem, pasta_destino):
    import csv
    import io
    import os
    import tempfile

    pasta_destino = Path(pasta_destino)
    pasta_backup = pasta_destino.parent / "EcoData_Duplicados_Arquivados"
    categorias = ["Chuvas", "Temperatura", "Qualidade_do_Ar", "Pendencias"]
    contagem = {categoria: 0 for categoria in categorias}
    ignorados = 0
    conflitos = 0
    migracoes = 0
    erros = 0

    for categoria in categorias:
        (pasta_destino / categoria).mkdir(parents=True, exist_ok=True)

    # Representa os campos CSV de forma estável entre LF e CRLF.
    # Mantém diferenças nos textos, na ordem das linhas e nos identificadores.
    def assinatura_csv(arquivo):
        buffer = io.StringIO(newline="")
        with Path(arquivo).open("r", encoding="utf-8-sig", newline="") as entrada:
            leitor = csv.reader(entrada, strict=True)
            cabecalho = next(leitor, None)
            if not cabecalho:
                raise ValueError("CSV vazio ou sem cabeçalho")
            escritor = csv.writer(buffer, lineterminator="\n")
            escritor.writerow(cabecalho)
            for linha in leitor:
                escritor.writerow(linha)
        return hashlib.sha256(buffer.getvalue().encode("utf-8")).hexdigest()

    def possivel_migracao_r5(esperado, existente):
        # Apenas sinaliza equivalência numérica de IDs textuais divergentes.
        # Não converte arquivos históricos nem os declara equivalentes.
        from decimal import Decimal, InvalidOperation

        tabelas = []
        for caminho in (esperado, existente):
            with Path(caminho).open("r", encoding="utf-8-sig", newline="") as entrada:
                tabelas.append(list(csv.reader(entrada, strict=True)))
        novo, antigo = tabelas
        if not novo or not antigo or novo[0] != antigo[0] or len(novo) != len(antigo):
            return False
        ids = {
            indice for indice, nome in enumerate(novo[0])
            if nome in ("id_registro", "estacao_id")
        }
        mudou = False
        for linha_nova, linha_antiga in zip(novo[1:], antigo[1:]):
            if len(linha_nova) != len(novo[0]) or len(linha_antiga) != len(novo[0]):
                return False
            for indice, (atual, anterior) in enumerate(zip(linha_nova, linha_antiga)):
                if atual == anterior:
                    continue
                if indice not in ids:
                    return False
                try:
                    valor_atual, valor_anterior = Decimal(atual), Decimal(anterior)
                except InvalidOperation:
                    return False
                if not (valor_atual.is_finite() and valor_anterior.is_finite()
                        and valor_atual == valor_anterior):
                    return False
                mudou = True
        return mudou

    arquivos_invalidos = []

    def hashes_csv_validos(pasta):
        hashes = set()
        if pasta.is_dir():
            for arquivo_csv in pasta.rglob("*.csv"):
                try:
                    hashes.add(assinatura_csv(arquivo_csv))
                except Exception as erro:
                    arquivos_invalidos.append((arquivo_csv, erro))
        return hashes

    hashes_arquivados = hashes_csv_validos(pasta_backup)
    hashes_ativos = hashes_csv_validos(pasta_destino)
    erros += len(arquivos_invalidos)
    for arquivo_invalido, erro in arquivos_invalidos:
        print(f"ERRO ao comparar {arquivo_invalido}: {erro}")

    for arquivo in listar_arquivos(pasta_origem):
        categoria = identificar_categoria(arquivo)
        destino = pasta_destino / categoria / f"{arquivo.stem}.csv"
        temporario = None

        try:
            # O arquivo temporário fica no mesmo diretório do destino.
            # No Windows, ele precisa estar fechado antes da conversão.
            with tempfile.NamedTemporaryFile(
                prefix=".ecodata_", suffix=".tmp",
                dir=destino.parent, delete=False
            ) as arquivo_temporario:
                temporario = Path(arquivo_temporario.name)

            converter_para_csv(arquivo, temporario)
            assinatura = assinatura_csv(temporario)

            conflito = False
            motivo = "conteúdo divergente ou incompleto"
            if os.path.lexists(destino):
                try:
                    assinatura_destino = assinatura_csv(destino)
                except Exception as erro_destino:
                    conflito = True
                    motivo = f"destino inválido ou ilegível ({erro_destino})"
                else:
                    if assinatura_destino == assinatura:
                        hashes_ativos.add(assinatura)
                        ignorados += 1
                        print(f"IGNORADO (já existe e foi validado): {destino.name}")
                        continue
                    conflito = True

            # Reconhece uma saída alternativa criada em execução anterior.
            alternativos = sorted(set(
                list(destino.parent.glob(f"{destino.stem}_origem_*.csv")) +
                list(destino.parent.glob(f"{destino.stem}_recuperado*.csv"))
            ))
            ja_preservado = None
            for alternativo in alternativos:
                try:
                    if assinatura_csv(alternativo) == assinatura:
                        ja_preservado = alternativo
                        break
                except Exception:
                    continue

            if ja_preservado is not None:
                if conflito:
                    conflitos += 1
                    print(
                        f"CONFLITO: origem {arquivo.name}; destino {destino.name} "
                        f"({motivo}); conteúdo já preservado em "
                        f"{ja_preservado.name}. Verifique o destino conflitante."
                    )
                else:
                    print(
                        f"IGNORADO (já presente em nome alternativo): "
                        f"{arquivo.name} -> {ja_preservado.name}"
                    )
                hashes_ativos.add(assinatura)
                ignorados += 1
                continue

            # Preserva EC-02: um conteúdo arquivado só é ignorado
            # quando outra cópia válida já está ativa.
            if assinatura in hashes_arquivados and assinatura in hashes_ativos:
                if conflito:
                    conflitos += 1
                    print(
                        f"CONFLITO: origem {arquivo.name}; destino {destino.name} "
                        f"({motivo}). Verifique o destino conflitante."
                    )
                ignorados += 1
                print(f"IGNORADO (conteúdo já presente na base ativa): {arquivo.name}")
                continue

            # Se a classificação mudou, preserve a cópia ativa anterior
            # e sinalize a migração antes de criar outra cópia.
            copia_outra_categoria = None
            for categoria_anterior in categorias:
                if categoria_anterior == categoria:
                    continue
                anterior = pasta_destino / categoria_anterior / destino.name
                if not anterior.is_file():
                    continue
                try:
                    if assinatura_csv(anterior) == assinatura:
                        copia_outra_categoria = anterior
                        break
                except (OSError, UnicodeError, csv.Error, ValueError):
                    continue
            if copia_outra_categoria is not None:
                conflitos += 1
                print(
                    f"[MIGRAÇÃO DE CATEGORIA PENDENTE] Origem: {arquivo.name}; "
                    f"cópia ativa: {copia_outra_categoria}; "
                    f"categoria atual: {categoria}. "
                    "Nenhum arquivo foi publicado; revise a classificação "
                    "antes de migrar a cópia ativa."
                )
                continue

            # IDs preservados podem divergir de conversões históricas que
            # perderam zeros. Esses casos exigem migração autorizada, sem
            # publicar automaticamente uma segunda versão do mesmo conjunto.
            historicos = sorted(pasta_destino.rglob("*.csv"))
            if assinatura in hashes_ativos and pasta_backup.is_dir():
                historicos += sorted(pasta_backup.rglob("*.csv"))
            migracao_pendente = None
            for historico in historicos:
                try:
                    if possivel_migracao_r5(temporario, historico):
                        migracao_pendente = historico
                        break
                except (OSError, UnicodeError, csv.Error, ValueError):
                    continue
            if migracao_pendente is not None:
                migracoes += 1
                conflitos += 1
                print(
                    f"[MIGRAÇÃO R5 NECESSÁRIA] Origem: {arquivo.name}; "
                    f"cópia histórica: {migracao_pendente}. "
                    "Diferença restrita a identificadores numericamente equivalentes. "
                    "Confirme os originais antes de autorizar a migração; "
                    "nenhum arquivo foi publicado ou substituído."
                )
                continue

            if conflito:
                if assinatura in hashes_arquivados:
                    nome_alternativo = f"{destino.stem}_recuperado"
                else:
                    nome_alternativo = (
                        f"{destino.stem}_origem_{arquivo.suffix.lower().lstrip('.')}"
                    )
                candidato = destino.with_name(f"{nome_alternativo}.csv")
            else:
                nome_alternativo = None
                candidato = destino
            numero = 2

            while True:
                if os.path.lexists(candidato):
                    try:
                        assinatura_candidato = assinatura_csv(candidato)
                    except Exception:
                        assinatura_candidato = None
                    if assinatura_candidato == assinatura:
                        hashes_ativos.add(assinatura)
                        ignorados += 1
                        if candidato != destino:
                            conflitos += 1
                            print(
                                f"CONFLITO: origem {arquivo.name}; destino "
                                f"{destino.name} ({motivo}); conteúdo já preservado "
                                f"em {candidato.name}. Verifique o destino conflitante."
                            )
                        else:
                            print(f"IGNORADO (já existe e foi validado): {destino.name}")
                        break
                    if candidato == destino:
                        conflito = True
                        motivo = "conteúdo divergente ou incompleto"
                        nome_alternativo = (
                            f"{destino.stem}_origem_{arquivo.suffix.lower().lstrip('.')}"
                        )
                        candidato = destino.with_name(f"{nome_alternativo}.csv")
                    else:
                        candidato = destino.with_name(
                            f"{nome_alternativo}_{numero}.csv"
                        )
                        numero += 1
                    continue

                try:
                    # Um hard link no mesmo volume publica sem substituir
                    # um destino criado por outra execução.
                    os.link(temporario, candidato)
                except FileExistsError:
                    continue

                hashes_ativos.add(assinatura)
                contagem[categoria] += 1
                if conflito:
                    conflitos += 1
                    print(
                        f"CONFLITO: origem {arquivo.name}; destino {destino.name} "
                        f"({motivo}); nova saída {categoria}/{candidato.name}. "
                        "Verifique o destino conflitante."
                    )
                else:
                    print(f"{arquivo.name}  ->  {categoria}/{candidato.name}")
                break

        except Exception as erro:
            erros += 1
            print(f"ERRO ao processar {arquivo.name}: {erro}")
        finally:
            if temporario is not None and temporario.exists():
                try:
                    temporario.unlink()
                except OSError as erro_limpeza:
                    erros += 1
                    print(f"ERRO ao remover temporário {temporario.name}: {erro_limpeza}")

    print("\nRESUMO DA ORGANIZAÇÃO")
    for categoria, total in contagem.items():
        print(f"{categoria}: {total}")
    print(f"Ignorados (existentes ou redundantes): {ignorados}")
    print(f"Conflitos: {conflitos}")
    print(f"Migrações R5 pendentes: {migracoes}")
    print(f"Erros: {erros}")


# ==================================================
# EXECUÇÃO
# ==================================================

organizar_arquivos(
    pasta_origem,
    pasta_destino
)


IGNORADO (já existe e foi validado): ar_copacabana_2026-10-18.csv
IGNORADO (conteúdo já presente na base ativa): ar_ensaio_20261024_a_repasse.csv
IGNORADO (já existe e foi validado): arquivo_sem_identificacao.csv
IGNORADO (já existe e foi validado): chuva_bangu_2026-10-16.csv
IGNORADO (já existe e foi validado): chuva_ensaio_20261024_a.csv
IGNORADO (conteúdo já presente na base ativa): chuva_ensaio_20261024_b_repasse.csv
IGNORADO (já existe e foi validado): chuva_ensaio_20261024_c.csv
IGNORADO (já existe e foi validado): chuva_meier_2026-10-07.csv
IGNORADO (já existe e foi validado): chuva_tijuca_2026-10-03.csv
IGNORADO (já existe e foi validado): coleta_ensaio_20261024_a.csv
IGNORADO (já existe e foi validado): coleta_ensaio_20261024_c.csv
IGNORADO (já existe e foi validado): coleta_turno_b.csv
IGNORADO (conteúdo já presente na base ativa): dados_ensaio_20261024_b_repasse.csv
IGNORADO (já existe e foi validado): dados_estacao_04.csv
IGNORADO (conteúdo já presente na base ativa): lote_

## 5. Identificação de arquivos duplicados

`verificar_arquivos_duplicados()` compara o SHA-256 **binário** dos CSVs ativos, mesmo quando os nomes diferem. Informa quantos arquivos foram analisados, quantos são redundantes e quantos conteúdos únicos permanecem; não move nem exclui documentos. A comparação não detecta registros parcialmente repetidos entre arquivos.

No ensaio fictício de 24/10/2026, oito pares exatos foram identificados **antes** do tratamento. Depois que o programa arquivou as oito cópias redundantes, a saída salva desta célula registrou **38 arquivos analisados, zero duplicatas exatas e 38 conteúdos únicos**. Esses números descrevem aquele estado; uma nova execução deve refletir o inventário real.


In [32]:
# ==================================================
# VERIFICAÇÃO DE ARQUIVOS DUPLICADOS
# ==================================================

def verificar_arquivos_duplicados(pasta_destino):

    print("\nVERIFICAÇÃO DE ARQUIVOS DUPLICADOS")
    print("=" * 50)

    pasta = Path(pasta_destino)

    arquivos = sorted(
        pasta.rglob("*.csv")
    )

    hashes = {}
    duplicados = []

    for arquivo in arquivos:

        assinatura = calcular_hash(arquivo)

        if assinatura in hashes:

            arquivo_original = hashes[assinatura]

            duplicados.append(
                (arquivo_original, arquivo)
            )

            print("\nDUPLICATA ENCONTRADA")

            print(
                f"Arquivo original: "
                f"{arquivo_original.name}"
            )

            print(
                f"Arquivo duplicado: "
                f"{arquivo.name}"
            )

        else:

            hashes[assinatura] = arquivo

    print("\nRESUMO DA VERIFICAÇÃO")
    print("-" * 50)

    print(
        f"Arquivos analisados: {len(arquivos)}"
    )

    print(
        f"Arquivos duplicados: {len(duplicados)}"
    )

    print(
        f"Arquivos únicos: {len(hashes)}"
    )

    if not duplicados:
        print(
            "\nNenhum arquivo duplicado encontrado."
        )

    return duplicados


# ==================================================
# EXECUÇÃO
# ==================================================

arquivos_duplicados = verificar_arquivos_duplicados(
    pasta_destino
)



VERIFICAÇÃO DE ARQUIVOS DUPLICADOS

RESUMO DA VERIFICAÇÃO
--------------------------------------------------
Arquivos analisados: 38
Arquivos duplicados: 0
Arquivos únicos: 38

Nenhum arquivo duplicado encontrado.


## 6. Tratamento de arquivos duplicados

`remover_arquivos_duplicados(pasta_destino, executar=False)` está em **simulação** por padrão: a execução normal do notebook não arquiva nem remove arquivos. Com `executar=True` explícito, mantém uma cópia ativa, priorizando nomes sem `repasse`, e move a redundante para `EcoData_Duplicados_Arquivados`. Se já houver backup de hash idêntico e sua integridade for confirmada, remove somente a redundância ativa, sem criar outro backup. Os originais em `dados_recebidos` são preservados.

No ensaio de 24/10/2026, a **função do EcoData** arquivou oito novas cópias `repasse`; não foi uma ação automática do monitor. Somadas às duas já arquivadas, são **10 arquivos arquivados**. Restaram **38 ativos**, sem duplicatas exatas após reorganizar. A saída atualmente salva desta célula mostra uma nova **simulação**: 38 analisados, zero duplicatas encontradas e nenhum movimento. As contagens antigas de 22 ativos/dois arquivados pertencem apenas ao inventário histórico.


In [33]:
def remover_arquivos_duplicados(pasta_destino, executar=False):

    pasta_destino = Path(pasta_destino)

    if not pasta_destino.is_dir():
        raise FileNotFoundError(
            f"Pasta não encontrada: {pasta_destino}"
        )

    pasta_backup = (
        pasta_destino.parent / "EcoData_Duplicados_Arquivados"
    )

    arquivos = sorted(pasta_destino.rglob("*.csv"))

    grupos = {}
    duplicados = []
    backups_por_hash = {}

    # Verifica os arquivos que já estão arquivados
    if pasta_backup.is_dir():

        for arquivo_backup in sorted(pasta_backup.rglob("*.csv")):

            assinatura = calcular_hash(arquivo_backup)

            backups_por_hash.setdefault(
                assinatura,
                arquivo_backup
            )

    # Agrupa os arquivos organizados pelo conteúdo
    for arquivo in arquivos:

        assinatura = calcular_hash(arquivo)

        grupos.setdefault(
            assinatura,
            []
        ).append(arquivo)

    print("\nREMOÇÃO DE ARQUIVOS DUPLICADOS")
    print("=" * 50)

    for assinatura, grupo in grupos.items():

        if len(grupo) < 2:
            continue

        # Prioriza manter arquivos sem "repasse"
        grupo.sort(
            key=lambda arquivo: (
                "repasse" in arquivo.stem.lower(),
                str(arquivo).lower()
            )
        )

        arquivo_mantido = grupo[0]

        for arquivo_duplicado in grupo[1:]:

            duplicados.append(arquivo_duplicado)

            print(f"\nArquivo mantido: {arquivo_mantido.name}")
            print(f"Duplicata: {arquivo_duplicado.name}")

            # Verifica se o conteúdo já existe no backup
            if assinatura in backups_por_hash:

                arquivo_no_backup = backups_por_hash[assinatura]

                if executar:

                    # Confirma a integridade do backup
                    if (
                        not arquivo_no_backup.is_file()
                        or calcular_hash(arquivo_no_backup) != assinatura
                    ):
                        raise RuntimeError(
                            f"Backup não pôde ser confirmado: {arquivo_no_backup}"
                        )

                    # Remove somente a cópia redundante
                    # da pasta EcoData_Organizado
                    arquivo_duplicado.unlink()

                    print(
                        "Status: backup idêntico já existe; "
                        "cópia redundante removida da pasta organizada"
                    )

                else:

                    print(
                        "Status: backup idêntico já existe; "
                        "nenhuma alteração (simulação)"
                    )

                continue

            # Simulação sem modificar arquivos
            if not executar:

                print(
                    "Status: identificado, sem alterações"
                )

                continue

            # Arquiva apenas quando ainda não existe
            # uma cópia idêntica no backup
            destino_backup = (
                pasta_backup /
                arquivo_duplicado.relative_to(pasta_destino)
            )

            destino_backup.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            destino_final = destino_backup
            contador = 2

            # Evita sobrescrever nomes já existentes
            while destino_final.exists():

                destino_final = destino_backup.with_name(
                    f"{destino_backup.stem}_{contador}"
                    f"{destino_backup.suffix}"
                )

                contador += 1

            shutil.move(
                str(arquivo_duplicado),
                str(destino_final)
            )

            # Atualiza o controle dos backups
            backups_por_hash[assinatura] = destino_final

            print("Status: movido para backup")

    print("\nRESUMO")
    print("-" * 50)

    print(f"Arquivos analisados: {len(arquivos)}")
    print(f"Duplicatas encontradas: {len(duplicados)}")

    if executar:

        restantes = len(
            list(pasta_destino.rglob("*.csv"))
        )

        print(f"Arquivos restantes: {restantes}")

    else:

        print(
            "Modo de simulação: "
            "nenhum arquivo foi movido ou removido"
        )

    return duplicados


# EXECUÇÃO DA FUNÇÃO

duplicados = remover_arquivos_duplicados(
    pasta_destino,
    executar=False # Selecionar True para remover arquivos duplicados ou False para simular a remoção 
)

duplicados_restantes = verificar_arquivos_duplicados(
    pasta_destino
)

# Apresenta a mensagem conforme o resultado
if len(duplicados_restantes) == 0:

    print("\nSUCESSO!")
    print("A base EcoData está livre de arquivos duplicados exatos.")

else:

    print("\nATENÇÃO!")
    print(f"Foram encontrados {len(duplicados_restantes)} arquivos duplicados.")


REMOÇÃO DE ARQUIVOS DUPLICADOS

RESUMO
--------------------------------------------------
Arquivos analisados: 38
Duplicatas encontradas: 0
Modo de simulação: nenhum arquivo foi movido ou removido

VERIFICAÇÃO DE ARQUIVOS DUPLICADOS

RESUMO DA VERIFICAÇÃO
--------------------------------------------------
Arquivos analisados: 38
Arquivos duplicados: 0
Arquivos únicos: 38

Nenhum arquivo duplicado encontrado.

SUCESSO!
A base EcoData está livre de arquivos duplicados exatos.


## 7. Inspeção inicial da qualidade dos dados

`verificar_qualidade()` mostra nome, categoria, quantidade de registros, colunas, valores ausentes e linhas exatamente repetidas **dentro de cada CSV**; não compara registros entre arquivos nem modifica dados.

No inventário observado em 24/10/2026, os **38 CSVs ativos** somam **738 registros**. A leitura não encontrou linhas internamente repetidas. Esse diagnóstico não substitui a análise de sobreposição parcial entre bases.


In [34]:
def verificar_qualidade(pasta_destino):
    pasta_destino = Path(pasta_destino)

    print("\nRELATÓRIO INICIAL DE QUALIDADE")
    print("=" * 50)

    categorias = [
        "Chuvas",
        "Temperatura",
        "Qualidade_do_Ar",
        "Pendencias"
    ]

    for categoria in categorias:
        pasta = pasta_destino / categoria

        for arquivo in sorted(pasta.glob("*.csv")):
            try:
                dados = pd.read_csv(
                    arquivo, encoding="utf-8-sig",
                    dtype={"id_registro": "string", "estacao_id": "string"}
                )

                print(f"\nArquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Linhas: {len(dados)}")
                print(f"Colunas: {list(dados.columns)}")

                ausentes = dados.isna().sum().sum()
                duplicados = dados.duplicated().sum()

                print(f"Valores ausentes: {ausentes}")
                print(f"Linhas duplicadas: {duplicados}")

            except Exception as erro:
                print(f"Erro ao ler {arquivo.name}: {erro}")

verificar_qualidade(pasta_destino)



RELATÓRIO INICIAL DE QUALIDADE

Arquivo: chuva_bangu_2026-10-16.csv
Categoria: Chuvas
Linhas: 40
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: chuva_ensaio_20261024_a.csv
Categoria: Chuvas
Linhas: 3
Colunas: ['id_registro', 'estacao_id', 'data', 'precipitacao_mm', 'origem_dado']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: chuva_ensaio_20261024_c.csv
Categoria: Chuvas
Linhas: 3
Colunas: ['id_registro', 'estacao_id', 'data', 'precipitacao_mm', 'origem_dado']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: chuva_meier_2026-10-07.csv
Categoria: Chuvas
Linhas: 38
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 2
Linhas duplicadas: 0

Arquivo: chuva_tijuca_2026-10-03.csv
Categoria: Chuvas
Linhas: 32
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores aus

Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: precipitacao_centro_2026-10-10.csv
Categoria: Chuvas
Linhas: 28
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: precipitacao_madureira_2026-10-19.csv
Categoria: Chuvas
Linhas: 28
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: teste_automacao_chuva_20261024.csv
Categoria: Chuvas
Linhas: 3
Colunas: ['id_registro', 'estacao_id', 'data', 'precipitacao_mm', 'origem_dado']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: temp_bangu_2026-10-20.csv
Categoria: Temperatura
Linhas: 26
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'temperatura_c', 'umidade_relativa_pct']
Valores ausentes: 1
Linhas duplicadas: 0

Arquivo: temp_centro_2026-10-06.csv
Categoria: Temperatura
Linhas: 32
Colunas: ['id_registro', 'data_hora', 'estacao

## 8. Investigação dos valores ausentes

`detalhar_ausentes()` identifica arquivos, colunas e registros com `NaN`, sem preencher ou excluir valores. Erros de leitura são informados, contados e não interrompem os arquivos seguintes (R2).

| Arquivo | Coluna | Ausentes |
|---|---|---:|
| `chuva_meier_2026-10-07.csv` | `precipitacao_mm` | 2 |
| `temp_bangu_2026-10-20.csv` | `umidade_relativa_pct` | 1 |
| `pm25_tijuca_2026-10-11.csv` | `pm25_ug_m3` | 1 |

A saída salva após os testes registra **38 arquivos lidos, zero erros de leitura e quatro valores ausentes**. São as mesmas quatro ausências históricas; os CSVs fictícios acrescentados não trouxeram outras. Ausência não equivale a zero nem é automaticamente uma violação de limite.


In [35]:
def detalhar_ausentes(pasta_destino):
    pasta_destino = Path(pasta_destino)

    categorias = [
        "Chuvas",
        "Temperatura",
        "Qualidade_do_Ar",
        "Pendencias"
    ]

    print("\nINVESTIGAÇÃO DE VALORES AUSENTES")
    print("=" * 50)
    lidos = 0
    falhas = []

    for categoria in categorias:
        pasta = pasta_destino / categoria

        for arquivo in sorted(pasta.glob("*.csv")):
            try:
                dados = pd.read_csv(
                    arquivo, encoding="utf-8-sig",
                    dtype={"id_registro": "string", "estacao_id": "string"}
                )
            except UnicodeDecodeError as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE DECODIFICAÇÃO]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: confira o formato do original e descubra sua codificação.")
                print("Crie uma cópia em UTF-8-SIG sem alterar o original; confira estrutura e registros.")
                print("Reprocesse a cópia pelo fluxo seguro do EcoData e confira a categoria.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue
            except (pd.errors.ParserError, pd.errors.EmptyDataError, OSError) as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE LEITURA]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Tipo: {type(erro).__name__}")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: verifique a estrutura, o conteúdo e o acesso ao CSV.")
                print("Corrija uma cópia, valide sua leitura e reprocesse pelo fluxo seguro do EcoData.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue

            lidos += 1

            ausentes = dados.isna().sum()

            if ausentes.sum() > 0:
                print(f"\nArquivo: {arquivo.name}")

                print("\nColunas com valores ausentes:")
                print(ausentes[ausentes > 0])

                print("\nRegistros afetados:")
                registros = dados[
                    dados.isna().any(axis=1)
                ]

                print(registros.to_string(index=False))

    print("\nRESUMO DA LEITURA")
    print(f"Arquivos lidos com sucesso: {lidos}")
    print(f"Arquivos com erro de leitura: {len(falhas)}")
    if falhas:
        print("Arquivos não analisados:")
        for categoria_falha, caminho_falha in falhas:
            print(f"- {categoria_falha}: {caminho_falha}")

detalhar_ausentes(pasta_destino)



INVESTIGAÇÃO DE VALORES AUSENTES

Arquivo: chuva_meier_2026-10-07.csv

Colunas com valores ausentes:
precipitacao_mm    2
dtype: int64

Registros afetados:
        id_registro        data_hora estacao_id bairro  precipitacao_mm periodo_medicao
PLU-ME-20261007-009 2026-10-07 04:00  PLU-ME-01  Méier              NaN          30 min
PLU-ME-20261007-022 2026-10-07 10:30  PLU-ME-01  Méier              NaN          30 min



Arquivo: temp_bangu_2026-10-20.csv

Colunas com valores ausentes:
umidade_relativa_pct    1
dtype: int64

Registros afetados:
        id_registro        data_hora estacao_id bairro  temperatura_c  umidade_relativa_pct
TMP-BG-20261020-013 2026-10-20 06:00  TMP-BG-01  Bangu           32.5                   NaN

Arquivo: pm25_tijuca_2026-10-11.csv

Colunas com valores ausentes:
pm25_ug_m3    1
dtype: int64

Registros afetados:
       id_registro        data_hora estacao_id bairro  pm25_ug_m3  pm10_ug_m3
AR-TJ-20261011-007 2026-10-11 03:00   AR-TJ-01 Tijuca         NaN        60.3

RESUMO DA LEITURA
Arquivos lidos com sucesso: 38
Arquivos com erro de leitura: 0


## 9. Validação das informações ambientais

`validar_valores()` verifica não numéricos e as regras abaixo, sem corrigir ou excluir medições. Erros de leitura são mostrados e a análise continua (R2). Se `id_registro` não existir ou estiver vazio, a ocorrência é localizada pelo índice do DataFrame (R3), que não é necessariamente a linha física do CSV.

| Coluna | Regra |
|---|---|
| `precipitacao_mm` | Não negativa |
| `umidade_relativa_pct` | De 0 a 100 |
| `pm25_ug_m3`, `pm10_ug_m3` | Não negativas |

A saída salva registra **30 CSVs lidos e zero erros de leitura** nas três categorias com regras ambientais; os **oito arquivos de Pendencias** não entram nessa validação. Não foram mostradas violações dos limites ou valores não numéricos nas colunas verificadas. Isso não comprova qualidade integral da base.


In [36]:
def validar_valores(pasta_destino):
    pasta_destino = Path(pasta_destino)

    regras = {
        "Chuvas": {
            "precipitacao_mm": (0, None)
        },
        "Temperatura": {
            "umidade_relativa_pct": (0, 100)
        },
        "Qualidade_do_Ar": {
            "pm25_ug_m3": (0, None),
            "pm10_ug_m3": (0, None)
        }
    }

    print("\nVALIDAÇÃO DE VALORES")
    print("=" * 50)
    lidos = 0
    falhas = []

    for categoria, colunas in regras.items():
        pasta = pasta_destino / categoria

        for arquivo in sorted(pasta.glob("*.csv")):
            try:
                dados = pd.read_csv(
                    arquivo, encoding="utf-8-sig",
                    dtype={"id_registro": "string", "estacao_id": "string"}
                )
            except UnicodeDecodeError as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE DECODIFICAÇÃO]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: confira o formato do original e descubra sua codificação.")
                print("Crie uma cópia em UTF-8-SIG sem alterar o original; confira estrutura e registros.")
                print("Reprocesse a cópia pelo fluxo seguro do EcoData e confira a categoria.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue
            except (pd.errors.ParserError, pd.errors.EmptyDataError, OSError) as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE LEITURA]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Tipo: {type(erro).__name__}")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: verifique a estrutura, o conteúdo e o acesso ao CSV.")
                print("Corrija uma cópia, valide sua leitura e reprocesse pelo fluxo seguro do EcoData.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue

            lidos += 1

            for coluna, limites in colunas.items():
                if coluna not in dados.columns:
                    print(f"{arquivo.name}: coluna {coluna} ausente")
                    continue

                minimo, maximo = limites

                valores = pd.to_numeric(
                    dados[coluna],
                    errors="coerce"
                )

                nao_numericos = (
                    dados[coluna].notna() &
                    valores.isna()
                )

                invalidos = valores < minimo

                if maximo is not None:
                    invalidos = invalidos | (valores > maximo)

                problemas = invalidos | nao_numericos

                if problemas.any():
                    print(f"\nArquivo: {arquivo.name}")
                    print(f"Categoria: {categoria}")
                    print(f"Coluna: {coluna}")

                    print(f"Valores problemáticos: {problemas.sum()}")

                    if "id_registro" not in dados.columns:
                        print("\n[AVISO] COLUNA IDENTIFICADORA AUSENTE")
                        print("A coluna 'id_registro' não foi encontrada.")
                        print("Os registros problemáticos serão localizados pelo índice do DataFrame.")
                        print("Nenhum identificador foi criado e nenhuma medição foi alterada.")
                        registros = dados.loc[problemas, [coluna]].copy()
                        registros.index.name = "Índice do DataFrame"
                        print(registros.to_string(index=True))
                    else:
                        registros = dados.loc[
                            problemas,
                            ["id_registro", coluna]
                        ].copy()
                        id_sem_valor = (
                            registros["id_registro"].isna()
                            | registros["id_registro"]
                            .astype("string")
                            .str.strip()
                            .eq("")
                            .fillna(False)
                        )

                        if id_sem_valor.any():
                            print("\n[AVISO] IDENTIFICADOR INDISPONÍVEL")
                            print("Um ou mais valores de 'id_registro' estão nulos ou vazios.")
                            print(f"Identificadores indisponíveis: {id_sem_valor.sum()}")
                            print("O índice do DataFrame identifica essas ocorrências; nenhum ID foi alterado.")
                            registros.index.name = "Índice do DataFrame"
                            print(registros.to_string(index=True))
                        else:
                            print(registros.to_string(index=False))

    print("\nRESUMO DA LEITURA")
    print(f"Arquivos lidos com sucesso: {lidos}")
    print(f"Arquivos com erro de leitura: {len(falhas)}")
    if falhas:
        print("Arquivos não analisados:")
        for categoria_falha, caminho_falha in falhas:
            print(f"- {categoria_falha}: {caminho_falha}")

validar_valores(pasta_destino)



VALIDAÇÃO DE VALORES

RESUMO DA LEITURA
Arquivos lidos com sucesso: 30
Arquivos com erro de leitura: 0


# EcoData | Referência técnica e inventário

## Objetivo e fluxo

O EcoData preserva os originais, organiza CSV/XLSX e investiga qualidade e duplicatas. A classificação usa tokens completos do nome; entradas desconhecidas ou ambíguas seguem para `Pendencias`. R1/EC-02 prepara a conversão em temporário, compara conteúdo e publica sem sobrescrever. Conflitos são informados e preservados com nome alternativo; reexecuções reconhecem o conteúdo já ativo. R5 mantém `id_registro` e `estacao_id` como texto e sinaliza IDs Excel ambíguos ou possível migração histórica, sem adivinhar zeros. R6 evita realocação automática entre categorias. R2 informa falhas de leitura e continua; R3 usa o índice do DataFrame quando falta um identificador.

A organização resume publicações por categoria, ignorados, conflitos, migrações e erros. A inspeção de qualidade mostra estrutura, ausências e repetições internas; a validação examina precipitação, umidade, PM2,5 e PM10. Valores ausentes são investigados separadamente. Arquivos não lidos não são considerados aprovados. Nenhuma medição ou ID é corrigido automaticamente.

**Duplicatas:** a verificação usa SHA-256 dos bytes; a ingestão compara campos CSV com LF/CRLF normalizados, preservando texto e ordem. Não são critérios equivalentes nem detectam sobreposição parcial de registros. O notebook mantém `executar=False` na rotina de tratamento. O monitor `monitorar_ingestao.py` ainda procura `Automação_Ecodata.ipynb`, removido intencionalmente; por isso, não carrega o organizador até que seu caminho seja atualizado. Ele não arquiva duplicatas. Este RV01/RV02 é a cópia de execução; seu código Python coincidia com o do antigo principal na última comparação.

## Estado atual observado após os testes de 24/10/2026

Foram adicionados **20 CSVs fictícios**, cinco para cada tema e três registros por arquivo. Havia **oito pares de conteúdo binário idêntico com nomes diferentes** e quatro arquivos sem par, formando 12 novos conteúdos únicos. O organizador classificou as 20 entradas. Uma chamada **explícita da função EcoData com `executar=True`** arquivou oito cópias `repasse`; as duas cópias antigas já estavam arquivadas. Reexecutar a organização não recriou as duplicatas.

| Local | Arquivos | Registros CSV |
|---|---:|---:|
| `dados_recebidos` (originais CSV/XLSX) | **48** | — |
| Ativos — Chuvas | 10 | 208 |
| Ativos — Temperatura | 11 | 246 |
| Ativos — Qualidade_do_Ar | 9 | 164 |
| Ativos — Pendencias | 8 | 120 |
| **Total em `EcoData_Organizado`** | **38** | **738** |
| Arquivados — Chuvas | 3 | 38 |
| Arquivados — Temperatura | 2 | 6 |
| Arquivados — Qualidade_do_Ar | 3 | 38 |
| Arquivados — Pendencias | 2 | 6 |
| **Total em `EcoData_Duplicados_Arquivados`** | **10** | **88** |

As **saídas agora salvas** mostram: organização com 48 ignorados e zero publicações/conflitos/erros; verificação de 38 ativos e zero duplicatas exatas; tratamento em simulação com zero duplicatas; investigação de ausências com 38 leituras e quatro valores ausentes; validação ambiental com 30 leituras e nenhuma violação informada. A leitura independente dos 38 ativos confirmou zero linhas internamente repetidas e os quatro ausentes conhecidos. Os totais descrevem esta observação e podem mudar.

Dois arquivos com **nome exatamente igual** não coexistem na pasta plana `dados_recebidos`. Os pares usados para SHA-256 tinham nomes diferentes; a reexecução testou o mesmo nome entre origem e destino. Os originais e backups anteriores tiveram seus hashes preservados no ensaio.

## Histórico e limites

**Antes dos ensaios recentes**, a base inicial tinha 24 documentos/754 registros com redundâncias: Chuvas 7/228, Temperatura 7/234, Qualidade_do_Ar 6/184 e Pendencias 4/108. Seus pares exatos eram `chuva_tijuca_2026-10-03.csv` / `precipitacao_tijuca_2026-10-03_repasse.csv` e `poluicao_centro_2026-10-05_repasse.csv` / `qualidade_ar_centro_2026-10-05.csv`. Na conferência histórica de 24/10/2026 às 01:20:30 (-03:00), havia **22 ativos/690 registros e dois arquivados/64 registros**. Somar 690 + 64 reproduz 754 **com** redundâncias, não observações independentes. Os 22 ativos foram lidos sem erro e tinham hashes distintos; havia quatro ausentes, sem repetições internas ou violações das regras aplicáveis. `Pendencias` não recebe essas regras. **Esses números não representam o estado após os novos testes.**

R2/R3 vieram de `Automação_Ecodata_candidata_R1_R2_R3 copy.ipynb`, comparado ao commit `bc3623aec518199e77cd0b084bff2af561aeffad`; R4–R6 foram aplicados depois. Testes anteriores em cópias temporárias cobriram conflitos, leitura, índice sem ID, zeros/Excel ambíguo, classificação, recuperação sem cópia ativa e reexecuções. As 24 conversões do inventário então existente mantiveram os bytes anteriores; uma cópia com 22 ativos/dois arquivados ficou estável em três reexecuções. `testes_r2` e `testes_r3` já tinham sido removidas do diretório de trabalho antes desta atualização; não foram recriadas.

Limites: esquema textual explícito só para dois IDs; marcadores de ausência seguem o Pandas; XLSX usa a primeira aba. Não há validação geral de esquema ou qualidade científica, detecção de observações parcialmente repetidas ou reconstrução automática de IDs formatados. Além de exigir um processo ativo, o monitor precisa apontar para um notebook existente. Os testes não garantem ausência de outros defeitos. Revisões de migração, testes permanentes e novas regras ambientais dependem de autorização. `Versões Git` guarda referências históricas; confirme qual notebook está aberto antes de executar.
